# Model exploration: does segmentation before classification help? (4 other models)

Our own exploration, separate from the team's `pipeline_model/`. It follows the same process (fixed split, one notebook for every
dataset, choose on val, test once, paired tests) but uses **different models**, one per box in the project diagram:

| # | diagram box | model | tuned on val |
|---|---|---|---|
| 1 | Baseline (zero-shot) | SigLIP base, `google/siglip-base-patch16-224` | prompt template set |
| 2 | Classic | dense SIFT → bag of visual words + 2×2 spatial pyramid → χ² SVM | codebook size, `C` |
| 3 | Transfer learning | EfficientNet-B0, ImageNet weights | how much to unfreeze, lr |
| 4 | DNN / CNN | ResNet-18 from scratch | width, dropout |

**Datasets** (only `DATASET` changes between runs):

| dataset | preprocessing |
|---|---|
| `01_original` | resized originals (baseline) |
| `02_denoise_clahe` | 01 + wavelet denoise (BayesShrink) + CLAHE on the L channel of LAB |
| `03_denoise_clahe_gsam` | 02 + Grounded-SAM: food kept, grey background (124,116,104), cropped to the food |

01 → 02 measures denoise + CLAHE. **02 → 03 measures segmentation** with both held fixed: that is the research question.

**Data.** 10 classes × (578 train / 193 val / 192 test) = 9,630 images, the same images and split in every dataset (one manifest).
Never re-split: the folders are the split.

### How to run
- **Mac (VS Code / Jupyter):** pick the `model_exploration/.venv` kernel and *Run All*. A missing dataset is downloaded from the team Drive and unzipped into `data/dataset/`.
- **Colab:** *Runtime → Change runtime type → T4 GPU*, then *Run All*. Data goes to `/content/data/dataset`; results to `MyDrive/food_explore/results/`, so they survive a disconnect.
- Run `RUN_MODE = "smoke"` first (20 images per class, 2 epochs, smallest grids), then `"full"`.
- Everything trained is saved; a re-run loads it instead of training again (`RETRAIN = True` to force).

## 0. Configuration

### 0.1 Fair-comparison rules
1. **Same model, same search space, same budget on 01 / 02 / 03.** Each dataset may pick its own best value from the grid, chosen by val macro-F1.
2. **Same input pipeline everywhere:** pad to square with the grey fill → resize → normalise (ImageNet stats; SigLIP uses its own).
   Padding, not centre-crop, because many Grounded-SAM crops are wide and a centre crop would cut food off.
3. **No colour jitter**: colour and contrast are what dataset 02 changes. Augmentation is a mild random crop + horizontal flip, train only.
4. Same seed, same split. Test is never used to choose anything.
5. Change a setting below for every dataset or for none.

In [ ]:
# ==== the only cell you normally edit ====
DATASET = "01_original"     # "01_original" | "02_denoise_clahe" | "03_denoise_clahe_gsam"
RUN_MODE = "smoke"          # "smoke": 20 imgs/class/split, 2 epochs, small grids | "full": the real run
SEED = 42
RETRAIN = False             # True: ignore saved checkpoints / features / grid results and redo them

# ---- from here on: identical for every dataset ----
IMG_SIZE = 224
CACHE_SIZE = 256                    # images are padded to a square and kept in RAM at this size (crop/resize to IMG_SIZE from it)
PAD_FILL = (124, 116, 104)          # same grey as the Grounded-SAM background
BATCH_SIZE = 32
EVAL_BATCH_SIZE = 64
LABEL_SMOOTHING = 0.1

# 1. SigLIP zero-shot
SIGLIP_ID = "google/siglip-base-patch16-224"
TEMPLATE_SETS = {
    "classname": ["{}"],
    "a photo of": ["a photo of {}."],
    "food photo": ["a photo of {}, a type of food."],
    "ensemble": ["a photo of {}, a type of food.", "a close-up photo of {}.", "a photo of a plate of {}.",
                 "a good photo of {}.", "a cropped photo of {}."],
}

# 2. dense SIFT bag of visual words + chi2 SVM
SIFT_STEP = 8                       # grid spacing (px) on the IMG_SIZE grey image
SIFT_SCALES = (8, 16)               # keypoint diameters (px): each grid point is described at both scales
CODEBOOK_SAMPLE = 200               # descriptors sampled per TRAIN image to fit the k-means codebook
BOVW_K = [256, 512]                 # codebook sizes tried
BOVW_C = [0.1, 1, 10, 100]

# 3. EfficientNet-B0 transfer (ImageNet)
# unfreeze: "head" = classifier only | "top" = stages 6-7 + final 1x1 conv + classifier | "all"
EFFNET_GRID = [dict(unfreeze="head", lr=1e-3), dict(unfreeze="top", lr=3e-4), dict(unfreeze="all", lr=1e-4)]
EFFNET_TRAIN = dict(max_epochs=15, patience=4, weight_decay=1e-4)

# 4. ResNet-18 from scratch
RESNET18_GRID = [dict(base=b, dropout=d) for b in (32, 64) for d in (0.0, 0.3)]
RESNET18_TRAIN = dict(max_epochs=40, patience=6, lr=1e-3, weight_decay=5e-4)

EXPECTED_MANIFEST_SHA256 = "79eb1d1cf8a475b8deaa2a301acc64a1c3a334fd4a289c8d45af91c359c3c2a4"
DATASET_ZIP_IDS = {   # team Drive: dip-project/dataset_zips/
    "01_original": "1xjZA96aUu807Qho4iNX2wJj8_3W3tyYm",
    "02_denoise_clahe": "1BiYKjqN3qsjecZvrep0GaaC8gpZelyBF",
    "03_denoise_clahe_gsam": "1ZGpho6AFoR32VA4Wyf7vQwsE10ZviLq6",
}
SMOKE_PER_CLASS = 20
if RUN_MODE == "smoke":
    BOVW_K, BOVW_C = [64], [1, 10]
    EFFNET_GRID, RESNET18_GRID = EFFNET_GRID[:1], RESNET18_GRID[:1]
    EFFNET_TRAIN = {**EFFNET_TRAIN, "max_epochs": 2}
    RESNET18_TRAIN = {**RESNET18_TRAIN, "max_epochs": 2}

In [ ]:
# ==== paths (Mac or Colab) and dataset download ====
import os, sys, zipfile
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = Path("/content")
    OUT_ROOT = Path("/content/drive/MyDrive/food_explore/results")      # on Drive, so it survives a disconnect
else:
    here = Path.cwd().resolve()
    WORK = next((p for p in [here, *here.parents] if (p / "explore_models.ipynb").exists()), here)
    OUT_ROOT = WORK / "results"
    os.environ.setdefault("TORCH_HOME", str(WORK / "checkpoints" / "torch"))   # downloaded weights stay in the workspace
    os.environ.setdefault("HF_HOME", str(WORK / "checkpoints" / "hf"))
DATA_ROOT = WORK / "data" / "dataset"
META = DATA_ROOT / "meta"


def ensure_dataset(name):
    """Download <name>.zip from the team Drive (if needed) and unzip it into DATA_ROOT: <name>/... and meta/..."""
    if (DATA_ROOT / name).is_dir() and (META / "manifest.csv").exists():
        return
    zpath = DATA_ROOT / "_zips" / f"{name}.zip"
    if not zpath.exists():
        import gdown
        zpath.parent.mkdir(parents=True, exist_ok=True)
        gdown.download(id=DATASET_ZIP_IDS[name], output=str(zpath), quiet=False)
        # Drive quota error ("too many accesses")? Download the zip by hand into data/dataset/_zips/ and re-run.
    with zipfile.ZipFile(zpath) as z:
        z.extractall(DATA_ROOT)        # overwrites without asking; meta/manifest.csv is the same in every zip
    print("unzipped", zpath.name)


ensure_dataset(DATASET)
OUT = (OUT_ROOT / DATASET) if RUN_MODE == "full" else (OUT_ROOT / "smoke" / DATASET)
CKPT_DIR, FIG_DIR = OUT / "checkpoints", OUT / "figures"
for d in (OUT, CKPT_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("data   :", DATA_ROOT / DATASET)
print("results:", OUT)

In [ ]:
import json, time, random, hashlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import precision_recall_fscore_support, f1_score, average_precision_score, confusion_matrix
from sklearn.preprocessing import label_binarize
from IPython.display import display

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"                     # fp16 autocast on CUDA only
NON_BLOCKING = DEVICE.type == "cuda"
NUM_WORKERS = 0 if sys.platform == "darwin" else 2  # macOS Jupyter can't pickle notebook classes into workers (the RAM cache makes this cheap)
pd.set_option("display.precision", 4)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})


def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def savefig(fig, name):
    fig.savefig(FIG_DIR / f"{name}.png", dpi=120, bbox_inches="tight")


def finish(fig, title, name):
    """Lay out a grid figure with its title above the first row, save, show."""
    fig.tight_layout(rect=(0, 0, 1, 1 - 0.55 / fig.get_figheight()))
    fig.suptitle(title, y=1 - 0.12 / fig.get_figheight())
    savefig(fig, name)
    plt.show()


class Timer:
    """`with Timer("label"):` prints how long a block took, so slow steps on the Mac are easy to spot."""
    def __init__(self, label):
        self.label = label

    def __enter__(self):
        self.t0 = time.time()

    def __exit__(self, *exc):
        print(f"[time] {self.label}: {time.time() - self.t0:.1f}s")


seed_everything()
print(f"python {sys.version.split()[0]} | torch {torch.__version__} | torchvision {torchvision.__version__} | device {DEVICE}")

## 1. Data checks
Same checks as the team notebook: the manifest hash must match everyone's, every manifest image exists, folders agree with the manifest,
no image id appears in two splits, and every class has the same count per split.

In [ ]:
sha = hashlib.sha256((META / "manifest.csv").read_bytes()).hexdigest()
assert sha == EXPECTED_MANIFEST_SHA256, f"manifest differs from the team's ({sha}): wrong zip?"

df_all = pd.read_csv(META / "manifest.csv")
df_all["path"] = [str(DATA_ROOT / DATASET / r) for r in df_all.relpath]
missing = [p for p in df_all.path if not os.path.exists(p)]
assert not missing, f"{len(missing)} images in the manifest are missing, e.g. {missing[:3]}"
on_disk = {str(p.relative_to(DATA_ROOT / DATASET)) for p in (DATA_ROOT / DATASET).glob("*/*/*.jpg")}
assert on_disk == set(df_all.relpath), "folder contents and manifest differ"
assert (df_all.relpath.str.split("/").str[0] == df_all.split).all() and \
       (df_all.relpath.str.split("/").str[1] == df_all.label).all(), "folder and manifest split/label disagree"

CLASSES = sorted(df_all.label.unique())
K = len(CLASSES)
CLS2IDX = {c: i for i, c in enumerate(CLASSES)}
df_all["y"] = df_all.label.map(CLS2IDX)

key = df_all.label + "/" + df_all.relpath.str.split("/").str[-1]
assert not key.duplicated().any(), "the same image id appears in more than one split"

counts = df_all.pivot_table(index="label", columns="split", values="relpath", aggfunc="count")[["train", "val", "test"]]
counts.loc["total"] = counts.sum()
display(counts)
assert counts.drop("total").nunique().eq(1).all(), "classes are not balanced"

df = df_all.groupby(["split", "label"], group_keys=False).head(SMOKE_PER_CLASS) if RUN_MODE == "smoke" else df_all
splits = {s: df[df.split == s].reset_index(drop=True) for s in ("train", "val", "test")}
y_split = {s: v.y.values for s, v in splits.items()}
print({s: len(v) for s, v in splits.items()}, "images used in this run")

In [ ]:
removed = pd.read_csv(META / "removed.csv")
removed["label"] = removed.key.str.split("/").str[0]
print(f"{len(removed)} images were removed during cleaning (by the data engineer):")
display(pd.crosstab(removed.label, removed.reason, margins=True))

# segmentation quality of dataset 03, used later by the failure analysis (section 10)
GSAM_STATS = pd.read_csv(META / "gsam_stats.csv").set_index("relpath") if (META / "gsam_stats.csv").exists() else None
if GSAM_STATS is not None:
    print("Grounded-SAM fallback rate:", GSAM_STATS.groupby("split").fallback.mean().round(4).to_dict())

In [ ]:
# 6 random training images per class
rng = np.random.default_rng(SEED)
fig, axes = plt.subplots(K, 6, figsize=(12, 2 * K))
for i, c in enumerate(CLASSES):
    rows = df_all[(df_all.split == "train") & (df_all.label == c)]
    for j, p in enumerate(rng.choice(rows.path.values, 6, replace=False)):
        ax = axes[i, j]; ax.imshow(Image.open(p)); ax.set_xticks([]); ax.set_yticks([])
        if j == 0: ax.set_ylabel(c, rotation=0, ha="right", fontsize=9)
finish(fig, f"{DATASET}: train samples", "samples")

## 2. Input pipeline
Every image is padded to a square with the grey fill and resized to `CACHE_SIZE` **once**, then kept in RAM as uint8
(about 1.9 GB for the full 9,630 images). On the Mac, DataLoader workers can't be used from a notebook, so without this cache every
epoch would decode every JPEG again. Train: random crop (70–100% of the area) + flip → `IMG_SIZE`. Eval: resize → `IMG_SIZE`.

In [ ]:
IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]


def pad_to_square(img, fill=PAD_FILL):
    w, h = img.size
    s = max(w, h)
    out = Image.new("RGB", (s, s), fill)
    out.paste(img, ((s - w) // 2, (s - h) // 2))
    return out


def load_cached(path):
    return np.asarray(pad_to_square(Image.open(path).convert("RGB")).resize((CACHE_SIZE, CACHE_SIZE), Image.BICUBIC))


with Timer("RAM image cache"):
    IMAGES = {s: np.stack([load_cached(p) for p in f.path]) for s, f in splits.items()}   # split -> (N, S, S, 3) uint8
print({s: v.shape for s, v in IMAGES.items()}, f"{sum(v.nbytes for v in IMAGES.values()) / 1e9:.2f} GB")

train_tf = T.Compose([T.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0), ratio=(0.9, 1.1)), T.RandomHorizontalFlip(),
                      T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
eval_tf = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])


def show_tensor(t, mean=IMAGENET_MEAN, std=IMAGENET_STD):
    """Normalised CHW tensor -> HWC image in [0, 1] for plotting."""
    x = t.cpu() * torch.tensor(std)[:, None, None] + torch.tensor(mean)[:, None, None]
    return x.clamp(0, 1).permute(1, 2, 0).numpy()


class CachedDataset(Dataset):
    def __init__(self, split, transform):
        self.images, self.labels, self.transform = IMAGES[split], y_split[split], transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return self.transform(Image.fromarray(self.images[i])), int(self.labels[i])


def make_loader(split, transform, shuffle=False, batch_size=None):
    return DataLoader(CachedDataset(split, transform), batch_size=batch_size or (BATCH_SIZE if shuffle else EVAL_BATCH_SIZE),
                      shuffle=shuffle, num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda",
                      persistent_workers=NUM_WORKERS > 0)


# what the models see
demo = np.random.default_rng(SEED).choice(len(splits["train"]), 3, replace=False)
fig, axes = plt.subplots(3, 5, figsize=(12, 7.5))
for row, i in zip(axes, demo):
    img = Image.fromarray(IMAGES["train"][i])
    row[0].imshow(Image.open(splits["train"].path[i])); row[0].set_title("stored", fontsize=8)
    row[1].imshow(show_tensor(eval_tf(img))); row[1].set_title(f"eval {IMG_SIZE}x{IMG_SIZE}", fontsize=8)
    for ax in row[2:]:
        ax.imshow(show_tensor(train_tf(img))); ax.set_title("train aug", fontsize=8)
    for ax in row: ax.axis("off")
plt.tight_layout(); savefig(fig, "transforms"); plt.show()

## 3. Evaluation toolkit and training loop (shared by all models)
Adapted from the team notebook so the numbers are computed the same way: top-1/3/5, macro precision/recall/F1, per-class AP and mAP.
Every config tried is scored on val and test and saved to `all_models/<config>/` as soon as it is done; the chosen config is the one with
the best **val** macro-F1. Test scores of the other configs are for the report only.

In [ ]:
RESULTS = {}   # model name -> {"val": {...}, "test": {...}, "config": {...}}


def topk_accuracy(y, scores, k):
    top = np.argsort(-scores, axis=1)[:, :k]
    return float((top == y[:, None]).any(axis=1).mean())


def compute_metrics(y, scores):
    y = np.asarray(y)
    pred = scores.argmax(1)
    p, r, f1, _ = precision_recall_fscore_support(y, pred, average="macro", zero_division=0)
    Y = label_binarize(y, classes=np.arange(K))
    ap = np.array([average_precision_score(Y[:, c], scores[:, c]) for c in range(K)])
    pc_p, pc_r, pc_f, support = precision_recall_fscore_support(y, pred, labels=np.arange(K), zero_division=0)
    per_class = pd.DataFrame({"precision": pc_p, "recall": pc_r, "f1": pc_f, "AP": ap, "support": support}, index=CLASSES)
    summary = {"top1": topk_accuracy(y, scores, 1), "top3": topk_accuracy(y, scores, 3), "top5": topk_accuracy(y, scores, 5),
               "macro_precision": float(p), "macro_recall": float(r), "macro_f1": float(f1), "mAP": float(np.nanmean(ap)), "n": int(len(y))}
    return summary, per_class


def macro_f1(y, scores):
    return f1_score(y, scores.argmax(1), average="macro", zero_division=0)


def plot_confusion(y, pred, title, name):
    cm = confusion_matrix(y, pred, labels=np.arange(K))
    cmn = cm / cm.sum(1, keepdims=True).clip(min=1)
    fig, axes = plt.subplots(1, 2, figsize=(18, 7.5))
    for ax, m, fmt, sub in [(axes[0], cm, "d", "counts"), (axes[1], cmn, ".2f", "row-normalised (diagonal = recall)")]:
        im = ax.imshow(m, cmap="Blues")
        ax.set_xticks(range(K), CLASSES, rotation=45, ha="right"); ax.set_yticks(range(K), CLASSES); ax.grid(False)
        for i in range(K):
            for j in range(K):
                ax.text(j, i, format(m[i, j], fmt), ha="center", va="center", fontsize=8,
                        color="white" if m[i, j] > m.max() / 2 else "black")
        ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(f"{title}: {sub}")
        fig.colorbar(im, ax=ax, fraction=0.046)
    plt.tight_layout(); savefig(fig, f"{name}_confusion"); plt.show()


def save_predictions(name, split, y, scores):
    out = splits[split][["relpath", "label"]].copy()
    out["y_true"], out["y_pred"] = y, scores.argmax(1)
    out["pred_label"] = [CLASSES[i] for i in out.y_pred]
    for c in range(K):
        out[f"score_{CLASSES[c]}"] = scores[:, c]
    out.to_csv(OUT / name / f"predictions_{split}.csv", index=False)


def report_model(name, y_val, s_val, y_test, s_test, config):
    """Final evaluation of a chosen model: val + test metrics, per-class table, confusion matrix, saved predictions.
    Example images of right and wrong predictions are in the failure analysis (section 10)."""
    (OUT / name).mkdir(exist_ok=True)
    sv, _ = compute_metrics(y_val, s_val)
    st, per_class = compute_metrics(y_test, s_test)
    RESULTS[name] = {"val": sv, "test": st, "config": config}
    display(pd.DataFrame([sv, st], index=["val", "test"]))
    display(per_class)
    plot_confusion(y_test, s_test.argmax(1), f"{name} (test)", name)
    save_predictions(name, "val", y_val, s_val)
    save_predictions(name, "test", y_test, s_test)
    per_class.to_csv(OUT / name / "per_class_test.csv")
    with open(OUT / name / "metrics.json", "w") as f:
        json.dump({"dataset": DATASET, "model": name, "seed": SEED, "run_mode": RUN_MODE, **RESULTS[name]}, f, indent=2, default=str)


def save_final(name, kind, model, arch=None, metrics=None):
    """Save a model as results/<dataset>/<name>/model.{pt,joblib} + config.json.
    kind: "torch" (state_dict + arch), "siglip" (class text embeddings only) or "sklearn" (whole fitted object)."""
    import joblib
    d = OUT / name
    d.mkdir(parents=True, exist_ok=True)
    arch = json.loads(json.dumps(arch or {}, default=str))
    metrics = metrics or RESULTS.get(name, {})
    if kind == "sklearn":
        file = "model.joblib"
        joblib.dump(model, d / file)
    elif kind == "siglip":
        file = "model.pt"
        torch.save({"kind": kind, "arch": arch, "classes": CLASSES, "text_emb": model.text_emb.cpu()}, d / file)
    else:
        file = "model.pt"
        torch.save({"kind": kind, "arch": arch, "classes": CLASSES,
                    "state_dict": {k: v.cpu() for k, v in model.state_dict().items()}}, d / file)
    config = {"dataset": DATASET, "model": name, "kind": kind, "file": file, "arch": arch, "classes": CLASSES,
              "preprocessing": {"pad_to_square_fill": PAD_FILL, "cache_size": CACHE_SIZE, "img_size": IMG_SIZE},
              "seed": SEED, "run_mode": RUN_MODE, "torch": torch.__version__,
              "val_macro_f1": metrics.get("val", {}).get("macro_f1"),
              "test_macro_f1": metrics.get("test", {}).get("macro_f1"),
              "saved_at": time.strftime("%Y-%m-%d %H:%M:%S")}
    with open(d / "config.json", "w") as f:
        json.dump(config, f, indent=2, default=str)


ALL_DIR = OUT / "all_models"   # every config tried: model + config.json + metrics.json + scores.npz
ALL_DIR.mkdir(exist_ok=True)
CONFIGS, CHOSEN = {}, {}       # config name -> its val/test metrics; model family -> the config chosen on val


def config_done(name, arch):
    """Metrics of a config already trained, scored and saved with this arch, else None (then it is trained / scored now)."""
    p = ALL_DIR / name / "metrics.json"
    if RETRAIN or not p.exists() or not (ALL_DIR / name / "scores.npz").exists():
        return None
    res = json.loads(p.read_text())
    if res.get("arch") != json.loads(json.dumps(arch, default=str)):
        return None
    CONFIGS[name] = res
    return res


def record_config(name, family, kind, model, arch, scores, train_f1):
    """Score one tried config on val and test, save it to all_models/<name>/ and remember it.
    scores: {"val": (y, s), "test": (y, s)}. Test is recorded for the report; choosing uses val only."""
    arch = json.loads(json.dumps(arch, default=str))
    res = {"model": name, "family": family, "kind": kind, "arch": arch, "train_f1": float(train_f1),
           **{sp: compute_metrics(*scores[sp])[0] for sp in ("val", "test")}}
    save_final(f"all_models/{name}", kind, model, arch, metrics=res)
    np.savez(ALL_DIR / name / "scores.npz", **{f"{sp}_{k}": v for sp, (y, s) in scores.items() for k, v in (("y", y), ("s", s))})
    with open(ALL_DIR / name / "metrics.json", "w") as f:
        json.dump({"dataset": DATASET, **res}, f, indent=2, default=str)
    CONFIGS[name] = res
    return res


def config_scores(name):
    """(y_val, s_val, y_test, s_test) of a recorded config, read back from all_models/<name>/scores.npz."""
    z = np.load(ALL_DIR / name / "scores.npz")
    return z["val_y"], z["val_s"], z["test_y"], z["test_s"]

In [ ]:
# shared PyTorch training loop (EfficientNet-B0 and ResNet-18)
SCALER = torch.amp.GradScaler("cuda", enabled=USE_AMP)


def set_train_mode(model):
    """train() but keep every fully frozen part in eval mode: its BatchNorm running stats don't change, and its
    dropout / stochastic depth (EfficientNet has both inside each block) don't add noise to features that can't learn."""
    model.train()
    for m in model.modules():
        params = list(m.parameters())
        if params and not any(p.requires_grad for p in params):
            m.eval()


def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    set_train_mode(model) if training else model.eval()
    total, ys, ps = 0.0, [], []
    for x, y_cpu in loader:
        # non_blocking copies only on CUDA: on MPS they can deliver corrupted labels
        x, y = x.to(DEVICE, non_blocking=NON_BLOCKING), y_cpu.to(DEVICE, non_blocking=NON_BLOCKING)
        with torch.set_grad_enabled(training), torch.autocast(DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(x)
            loss = F.cross_entropy(logits, y, label_smoothing=LABEL_SMOOTHING if training else 0.0)
        if training:
            optimizer.zero_grad(set_to_none=True)
            SCALER.scale(loss).backward()
            SCALER.step(optimizer)
            SCALER.update()
        total += loss.item() * len(y)
        ys.append(y_cpu); ps.append(logits.detach().float().softmax(1).cpu())
    y, p = torch.cat(ys).numpy(), torch.cat(ps).numpy()
    return total / len(y), y, p


def fit(name, build, cfg, max_epochs, patience, lr, weight_decay):
    """Train with AdamW + cosine LR, keep the epoch with the best val macro-F1, stop after `patience` epochs without gain.
    Checkpoint files carry a hash of the config, so changing a grid value never reloads a stale model."""
    tag = hashlib.md5(str((cfg, max_epochs, patience, lr, weight_decay, LABEL_SMOOTHING, IMG_SIZE, CACHE_SIZE)).encode()).hexdigest()[:8]
    ckpt, hist_path = CKPT_DIR / f"{name}_{tag}.pt", CKPT_DIR / f"{name}_{tag}_history.csv"
    seed_everything()
    model = build().to(DEVICE)
    if ckpt.exists() and hist_path.exists() and not RETRAIN:
        model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
        print(f"{name}: loaded saved model {ckpt.name}")
        return model, pd.read_csv(hist_path), ckpt
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"{name}: {n_train / 1e6:.2f}M trainable params")
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max_epochs)
    train_dl, val_dl = make_loader("train", train_tf, shuffle=True), make_loader("val", eval_tf)
    rows, best, best_epoch, wait = [], -1.0, 0, 0
    for epoch in range(1, max_epochs + 1):
        t0, lr_now = time.time(), opt.param_groups[0]["lr"]
        tr_loss, tr_y, tr_p = run_epoch(model, train_dl, opt)
        va_loss, va_y, va_p = run_epoch(model, val_dl)
        sched.step()
        row = dict(epoch=epoch, lr=lr_now, train_loss=tr_loss, val_loss=va_loss,
                   train_acc=(tr_p.argmax(1) == tr_y).mean(), val_acc=(va_p.argmax(1) == va_y).mean(),
                   train_f1=macro_f1(tr_y, tr_p), val_f1=macro_f1(va_y, va_p), sec=time.time() - t0)
        rows.append(row)
        print(f"  ep {epoch:2d} | loss {tr_loss:.3f}/{va_loss:.3f} | F1 {row['train_f1']:.3f}/{row['val_f1']:.3f} | {row['sec']:.0f}s")
        if row["val_f1"] > best:
            best, best_epoch, wait = row["val_f1"], epoch, 0
            torch.save(model.state_dict(), ckpt)
        else:
            wait += 1
            if wait >= patience:
                print(f"  early stop: no val gain for {patience} epochs")
                break
    hist = pd.DataFrame(rows)
    hist["best"] = hist.epoch == best_epoch
    hist.to_csv(hist_path, index=False)
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    return model, hist, ckpt


@torch.no_grad()
def predict(model, split, transform=None):
    _, y, p = run_epoch(model, make_loader(split, transform or eval_tf))
    return y, p


def plot_learning_curves(hists, title, name):
    """One row per config: loss and macro-F1 per epoch, train vs val, dashed line at the kept epoch."""
    fig, axes = plt.subplots(len(hists), 2, figsize=(12, 3 * len(hists)), squeeze=False)
    for row, (cfg_name, h) in zip(axes, hists.items()):
        best = h.loc[h.best, "epoch"].iloc[0]
        for ax, metric in zip(row, ("loss", "f1")):
            ax.plot(h.epoch, h[f"train_{metric}"], "o-", ms=3, label="train")
            ax.plot(h.epoch, h[f"val_{metric}"], "o-", ms=3, label="val")
            ax.axvline(best, ls="--", c="gray", lw=1)
            ax.set_title(f"{cfg_name}: {'loss' if metric == 'loss' else 'macro-F1'} (kept epoch {best})", fontsize=10)
            ax.set_xlabel("epoch"); ax.legend()
            ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    finish(fig, title, name)


def plot_grid_summary(table, x, title, name, logx=False):
    """Train vs val macro-F1 for each config: the 'elbow' plot."""
    fig, ax = plt.subplots(figsize=(6.5, 4))
    xs = table[x] if logx else table[x].astype(str)
    ax.plot(xs, table.train_f1, "o-", label="train")
    ax.plot(xs, table.val_f1, "o-", label="val")
    if logx: ax.set_xscale("log")
    best = table.val_f1.idxmax()
    ax.scatter([xs[best]], [table.val_f1[best]], s=160, facecolors="none", edgecolors="red", lw=2, label="chosen (best val)", zorder=5)
    ax.set_xlabel(x); ax.set_ylabel("macro-F1"); ax.set_title(title); ax.legend()
    plt.tight_layout(); savefig(fig, name); plt.show()

## 4. Model 1: SigLIP zero-shot (baseline)
SigLIP (Zhai et al., ICCV 2023) is a CLIP-style image-text model trained with a **sigmoid** loss on each image-text pair instead of a softmax over the batch.
Zero-shot: each class becomes a text prompt; the image goes to the class whose text embedding is most similar. Nothing is trained, so this
row shows what preprocessing does to a model that has never seen our training images.
Only the prompt template set is chosen (on val). SigLIP's own normalisation is mean = std = 0.5.

In [ ]:
from transformers import AutoModel, AutoTokenizer

siglip = AutoModel.from_pretrained(SIGLIP_ID).to(DEVICE).eval()
for p in siglip.parameters():
    p.requires_grad_(False)
siglip_tok = AutoTokenizer.from_pretrained(SIGLIP_ID)
SIGLIP_MEAN = SIGLIP_STD = [0.5, 0.5, 0.5]
siglip_tf = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE), interpolation=T.InterpolationMode.BICUBIC),
                       T.ToTensor(), T.Normalize(SIGLIP_MEAN, SIGLIP_STD)])


def pooled(out):
    """transformers >= 5 returns an output object from get_*_features; older versions return the tensor itself."""
    return out.pooler_output if hasattr(out, "pooler_output") else out


@torch.no_grad()
def siglip_text_embeddings(templates):
    embs = []
    for c in CLASSES:
        t = siglip_tok([tpl.format(c.replace("_", " ")) for tpl in templates], padding="max_length", return_tensors="pt").to(DEVICE)
        e = pooled(siglip.get_text_features(input_ids=t["input_ids"]))       # SigLIP was trained with max_length padding
        embs.append(F.normalize(F.normalize(e, dim=-1).mean(0), dim=0))
    return torch.stack(embs)


class SigLIPZeroShot(nn.Module):
    """Image -> class logits: scaled cosine similarity to each class's text embedding (Grad-CAM-able, saved with its text_emb)."""
    def __init__(self, model, text_emb):
        super().__init__()
        self.model = model
        self.register_buffer("text_emb", text_emb)

    def forward(self, x):
        f = F.normalize(pooled(self.model.get_image_features(pixel_values=x)), dim=-1)
        return self.model.logit_scale.exp() * f @ self.text_emb.T     # SigLIP's logit_bias is the same for every class: no effect on the softmax


@torch.no_grad()
def siglip_image_embeddings(split):
    cache = CKPT_DIR / f"siglip_img_{split}.pt"
    if cache.exists() and not RETRAIN:
        z = torch.load(cache)
        if z["relpath"] == splits[split].relpath.tolist():
            return z["emb"].to(DEVICE)
    feats = []
    for x, _ in make_loader(split, siglip_tf):
        feats.append(F.normalize(pooled(siglip.get_image_features(pixel_values=x.to(DEVICE))), dim=-1))
    feats = torch.cat(feats)
    torch.save({"emb": feats.cpu(), "relpath": splits[split].relpath.tolist()}, cache)
    return feats


with Timer("SigLIP image embeddings (val + test)"):
    img_emb = {s: siglip_image_embeddings(s) for s in ("val", "test")}
scale = siglip.logit_scale.exp()
rows = []
for tname, templates in TEMPLATE_SETS.items():
    name, arch = "siglip_" + tname.replace(" ", "_"), {"siglip_id": SIGLIP_ID, "templates": templates}
    res = config_done(name, arch)
    if res is None:
        model = SigLIPZeroShot(siglip, siglip_text_embeddings(templates)).eval()
        sc = {sp: (y_split[sp], (scale * img_emb[sp] @ model.text_emb.T).softmax(1).cpu().numpy()) for sp in ("val", "test")}
        res = record_config(name, "siglip_zeroshot", "siglip", model, arch, sc, train_f1=float("nan"))   # nothing is trained
    rows.append({"templates": tname, "name": name, "val_top1": res["val"]["top1"], "val_f1": res["val"]["macro_f1"]})
siglip_grid = pd.DataFrame(rows)
display(siglip_grid)

fig, ax = plt.subplots(figsize=(6.5, 3.5))
ax.bar(siglip_grid.templates, siglip_grid.val_f1)
for i, v in enumerate(siglip_grid.val_f1): ax.text(i, v, f"{v:.3f}", ha="center", va="bottom")
ax.set_ylabel("val macro-F1"); ax.set_title("SigLIP zero-shot: prompt template (val)"); ax.set_ylim(0, 1)
plt.tight_layout(); savefig(fig, "siglip_templates"); plt.show()

best_siglip = siglip_grid.loc[siglip_grid.val_f1.idxmax()]
CHOSEN["siglip_zeroshot"] = best_siglip["name"]
siglip_model = SigLIPZeroShot(siglip, siglip_text_embeddings(TEMPLATE_SETS[best_siglip.templates])).eval()
print("chosen templates:", best_siglip.templates)
report_model("siglip_zeroshot", *config_scores(best_siglip["name"]), {"templates": TEMPLATE_SETS[best_siglip.templates]})
save_final("siglip_zeroshot", "siglip", siglip_model, {"siglip_id": SIGLIP_ID, "templates": TEMPLATE_SETS[best_siglip.templates]})

## 5. Model 2: classical, dense SIFT bag of visual words + χ² SVM
The standard pre-deep-learning recipe (Csurka et al. 2004; Lazebnik et al., CVPR 2006):
1. **Dense SIFT**: a SIFT descriptor (Lowe 2004) every `SIFT_STEP` px on the grey image, at each scale in `SIFT_SCALES`. A dense grid (not detected
   keypoints) because dark, low-texture food has few corners. Each descriptor is turned into **RootSIFT** (L1-normalise, square root; Arandjelović & Zisserman 2012).
2. **Codebook**: k-means on descriptors sampled from **train images only**; each centre is a "visual word".
3. **Spatial pyramid**: each image becomes a word histogram over the whole image (level 0) plus one per 2×2 quadrant (level 1), each level weighted ½.
4. **χ² kernel SVM** (Zhang et al., IJCV 2007): χ² compares histograms better than Euclidean distance. γ = 1 / mean χ² distance on train.

Tuned on val: codebook size `K` and `C`. Colour is not used, unlike the team's HSV+HOG+LBP; this model only sees grey texture and shape.
Extraction is CPU work, parallel over all cores.

In [ ]:
import cv2
from joblib import Parallel, delayed
from sklearn.cluster import MiniBatchKMeans
from sklearn.svm import SVC
from sklearn.metrics.pairwise import additive_chi2_kernel
import joblib


def dense_rootsift(img):
    """(S, S, 3) uint8 cache image -> RootSIFT descriptors (n, 128) and their (x, y) positions on the IMG_SIZE grey image."""
    gray = cv2.cvtColor(cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA), cv2.COLOR_RGB2GRAY)
    grid = range(SIFT_STEP, IMG_SIZE - SIFT_STEP + 1, SIFT_STEP)
    kps = [cv2.KeyPoint(float(x), float(y), float(s)) for s in SIFT_SCALES for y in grid for x in grid]
    kps, d = cv2.SIFT_create().compute(gray, kps)
    d = np.sqrt(d / (d.sum(1, keepdims=True) + 1e-7))
    return d.astype(np.float32), np.array([kp.pt for kp in kps], dtype=np.float32)


def sample_descriptors(imgs, seeds):
    out = []
    for img, seed in zip(imgs, seeds):
        d, _ = dense_rootsift(img)
        out.append(d[np.random.default_rng(seed).choice(len(d), min(CODEBOOK_SAMPLE, len(d)), replace=False)])
    return np.concatenate(out)


def assign(d, centres):
    dist = (d ** 2).sum(1)[:, None] - 2 * d @ centres.T + (centres ** 2).sum(1)[None]
    return dist.argmin(1)


def pyramid_histogram(words, xy, k):
    """Level 0 (whole image) + level 1 (2x2 quadrants), each level weighted 1/2 (Lazebnik et al. 2006, L = 1). Sums to 1."""
    half = IMG_SIZE / 2
    cell = (xy[:, 0] >= half).astype(int) + 2 * (xy[:, 1] >= half)
    hists = [0.5 * np.bincount(words, minlength=k)] + [0.5 * np.bincount(words[cell == c], minlength=k) for c in range(4)]
    return np.concatenate(hists).astype(np.float32) / len(words)


def bovw_chunk(imgs, codebooks):
    """Histograms of a chunk of images for every codebook at once, so SIFT runs once per image."""
    out = {k: [] for k in codebooks}
    for img in imgs:
        d, xy = dense_rootsift(img)
        for k, centres in codebooks.items():
            out[k].append(pyramid_histogram(assign(d, centres), xy, k))
    return {k: np.stack(v) for k, v in out.items()}


def chunks(n, size=64):
    return [slice(i, min(i + size, n)) for i in range(0, n, size)]


BOVW_TAG = hashlib.md5(str((SIFT_STEP, SIFT_SCALES, CODEBOOK_SAMPLE, IMG_SIZE, CACHE_SIZE, SEED)).encode()).hexdigest()[:8]

# 1-2. codebooks, fit on TRAIN descriptors only
codebooks = {}
for k in BOVW_K:
    path = CKPT_DIR / f"codebook_K{k}_{BOVW_TAG}.npz"
    if path.exists() and not RETRAIN:
        z = np.load(path, allow_pickle=True)
        if list(z["relpath"]) == splits["train"].relpath.tolist():
            codebooks[k] = z["centres"]
if len(codebooks) < len(BOVW_K):
    with Timer("SIFT sample for the codebook (train only)"):
        imgs = IMAGES["train"]
        parts = Parallel(n_jobs=-1)(delayed(sample_descriptors)(imgs[s], range(s.start, s.stop)) for s in chunks(len(imgs)))
        sample = np.concatenate(parts)
    print(f"codebook sample: {sample.shape[0]:,} descriptors from {len(imgs)} train images")
    assert len(imgs) == len(splits["train"]), "the codebook must come from the train split only"
    for k in BOVW_K:
        if k in codebooks:
            continue
        with Timer(f"k-means K={k}"):
            km = MiniBatchKMeans(n_clusters=k, batch_size=4096, n_init=3, random_state=SEED).fit(sample)
        codebooks[k] = km.cluster_centers_.astype(np.float32)
        np.savez(CKPT_DIR / f"codebook_K{k}_{BOVW_TAG}.npz", centres=codebooks[k], relpath=splits["train"].relpath.values)
    del sample, parts

# 3. pyramid histograms for every split and codebook
def load_bovw(split):
    paths = {k: CKPT_DIR / f"bovw_K{k}_{BOVW_TAG}_{split}.npz" for k in BOVW_K}
    feats = {}
    for k, p in paths.items():
        if p.exists() and not RETRAIN:
            z = np.load(p, allow_pickle=True)
            if list(z["relpath"]) == splits[split].relpath.tolist():
                feats[k] = z["X"]
    todo = {k: codebooks[k] for k in BOVW_K if k not in feats}
    if todo:
        with Timer(f"dense SIFT + histograms, {split}"):
            imgs = IMAGES[split]
            parts = Parallel(n_jobs=-1)(delayed(bovw_chunk)(imgs[s], todo) for s in chunks(len(imgs)))
        for k in todo:
            feats[k] = np.concatenate([p[k] for p in parts])
            np.savez(paths[k], X=feats[k], relpath=splits[split].relpath.values)
    return feats


BOVW = {s: load_bovw(s) for s in ("train", "val", "test")}   # split -> {K: (n, 5K) histograms}
print({s: {k: v.shape for k, v in f.items()} for s, f in BOVW.items()})

In [ ]:
def chi2_block(a, b):
    # joblib hands large arrays to workers as read-only memory maps; sklearn's Cython chi2 code needs writable arrays
    return additive_chi2_kernel(np.array(a), np.array(b))


def chi2_distance(A, B):
    """Pairwise chi2 distance sum (a-b)^2 / (a+b), computed in parallel row blocks."""
    parts = Parallel(n_jobs=-1)(delayed(chi2_block)(A[s], B) for s in chunks(len(A), 256))
    return -np.concatenate(parts)


rows = []
for k in BOVW_K:
    Xtr, Xva, Xte = BOVW["train"][k], BOVW["val"][k], BOVW["test"][k]
    D = None
    for c in BOVW_C:
        name = f"bovw_K{k}_C{c:g}"
        arch = {"K": k, "C": c, "sift_step": SIFT_STEP, "sift_scales": list(SIFT_SCALES), "codebook_sample": CODEBOOK_SAMPLE, "kernel": "chi2"}
        res = config_done(name, arch)
        if res is None:
            if D is None:   # kernels once per codebook, shared by every C
                with Timer(f"chi2 kernels K={k}"):
                    D = {"train": chi2_distance(Xtr, Xtr), "val": chi2_distance(Xva, Xtr), "test": chi2_distance(Xte, Xtr)}
                gamma = 1.0 / D["train"].mean()
                Kmat = {sp: np.exp(-gamma * d) for sp, d in D.items()}
            t0 = time.time()
            svm = SVC(kernel="precomputed", C=c, decision_function_shape="ovr").fit(Kmat["train"], y_split["train"])
            model = {"svm": svm, "codebook": codebooks[k], "gamma": gamma, "train_features": f"bovw_K{k}_{BOVW_TAG}_train.npz"}
            res = record_config(name, "bovw_svm", "sklearn", model, arch,
                                {sp: (y_split[sp], svm.decision_function(Kmat[sp])) for sp in ("val", "test")},
                                train_f1=macro_f1(y_split["train"], svm.decision_function(Kmat["train"])))
            print(f"  K={k} C={c:g} -> train {res['train_f1']:.3f} | val {res['val']['macro_f1']:.3f} | {time.time() - t0:.0f}s")
        rows.append({"K": k, "C": c, "name": name, "train_f1": res["train_f1"], "val_f1": res["val"]["macro_f1"]})
    D = Kmat = None
bovw_grid = pd.DataFrame(rows)
display(bovw_grid)

fig, ax = plt.subplots(figsize=(6.5, 4))
for j, (k, g) in enumerate(bovw_grid.groupby("K")):
    ax.plot(g.C, g.train_f1, "o--", c=f"C{j}", alpha=0.6, label=f"K={k} train")
    ax.plot(g.C, g.val_f1, "o-", c=f"C{j}", label=f"K={k} val")
best = bovw_grid.loc[bovw_grid.val_f1.idxmax()]
ax.scatter([best.C], [best.val_f1], s=160, facecolors="none", edgecolors="red", lw=2, label="chosen (best val)", zorder=5)
ax.set_xscale("log"); ax.set_xlabel("C"); ax.set_ylabel("macro-F1"); ax.set_title("SIFT BoVW + chi2 SVM: validation curve"); ax.legend(fontsize=8)
plt.tight_layout(); savefig(fig, "bovw_curve"); plt.show()

CHOSEN["bovw_svm"] = best["name"]
print("chosen:", best["name"])
report_model("bovw_svm", *config_scores(best["name"]), {"K": int(best.K), "C": float(best.C)})
save_final("bovw_svm", "sklearn", joblib.load(ALL_DIR / best["name"] / "model.joblib"), {"K": int(best.K), "C": float(best.C)})

## 6. Model 3: EfficientNet-B0 transfer learning (ImageNet)
EfficientNet-B0 (Tan & Le, ICML 2019): a CNN found by architecture search and scaled evenly in depth, width and resolution. It reaches
ResNet-50-level ImageNet accuracy with about 5M parameters (ResNet-50 has 25M), so it is cheap to fine-tune on a Mac.
Tuned on val: how much of the network is trained (`head` only / `top` stages / `all`), each with its own lr (smaller when more layers move,
so pretrained features are not destroyed; Kornblith et al., CVPR 2019 compare fixed features with fine-tuning).
Frozen parts stay in eval mode (BatchNorm statistics, dropout and stochastic depth unchanged).

In [ ]:
def make_effnet(unfreeze, pretrained=True):
    m = torchvision.models.efficientnet_b0(weights=torchvision.models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None)
    m.classifier = nn.Sequential(nn.Dropout(0.2), nn.Linear(m.classifier[1].in_features, K))
    for p in m.parameters():
        p.requires_grad_(False)
    parts = {"head": [m.classifier], "top": [m.features[6], m.features[7], m.features[8], m.classifier], "all": [m]}[unfreeze]
    for part in parts:
        for p in part.parameters():
            p.requires_grad_(True)
    return m


en_hists, en_rows = {}, []
for cfg in EFFNET_GRID:
    name = f"effnet_b0_{cfg['unfreeze']}"
    model, hist, ckpt = fit(name, lambda: make_effnet(cfg["unfreeze"]), cfg, lr=cfg["lr"], **EFFNET_TRAIN)
    en_hists[name] = hist
    kept = hist[hist.best].iloc[0]
    if config_done(name, cfg) is None or ckpt.stat().st_mtime > (ALL_DIR / name / "metrics.json").stat().st_mtime:   # new or retrained
        record_config(name, "effnet_b0", "torch", model.eval(), cfg, {sp: predict(model, sp) for sp in ("val", "test")}, kept.train_f1)
    en_rows.append({**cfg, "name": name, "epoch": int(kept.epoch), "epochs_run": len(hist),
                    "stopped": "early stop" if len(hist) < EFFNET_TRAIN["max_epochs"] else "epoch limit",
                    "train_f1": kept.train_f1, "val_f1": kept.val_f1, "sec_per_epoch": hist.sec.mean(), "ckpt": str(ckpt)})
    del model
en_grid = pd.DataFrame(en_rows)
display(en_grid.drop(columns="ckpt"))
plot_learning_curves(en_hists, "EfficientNet-B0 transfer: learning curves", "effnet_learning_curves")
plot_grid_summary(en_grid, "unfreeze", "EfficientNet-B0: train vs val macro-F1 by fine-tuned depth", "effnet_unfreeze")

best_en = en_grid.loc[en_grid.val_f1.idxmax()]
CHOSEN["effnet_b0"] = best_en["name"]
en_model = make_effnet(best_en.unfreeze).to(DEVICE)
en_model.load_state_dict(torch.load(best_en.ckpt, map_location=DEVICE)); en_model.eval()
print("chosen:", best_en["name"])
report_model("effnet_b0", *config_scores(best_en["name"]), {"unfreeze": best_en.unfreeze, "lr": best_en.lr, "epoch": int(best_en.epoch)})
save_final("effnet_b0", "torch", en_model, {"arch": "efficientnet_b0", "unfreeze": best_en.unfreeze})

## 7. Model 4: ResNet-18 trained from scratch
ResNet (He et al., CVPR 2016) adds a **skip connection** around every pair of conv layers, so each block learns a correction to its input;
this lets deep networks train from random weights. The team's CNN has no skip connections, so this asks whether residual learning helps
when there is no pretraining and only 578 images per class.
Same layout as torchvision's ResNet-18 (4 stages × 2 basic blocks), with the stage widths scaled by `base` (64 = standard, 32 = slim, ¼ of
the parameters). Tuned on val: `base` and dropout before the classifier.

In [ ]:
from torchvision.models.resnet import BasicBlock


class ResNet18(nn.Module):
    def __init__(self, base=64, dropout=0.0, n_classes=K):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, base, 7, 2, 3, bias=False), nn.BatchNorm2d(base), nn.ReLU(inplace=True), nn.MaxPool2d(3, 2, 1))
        c_in = base
        for i, c in enumerate([base, 2 * base, 4 * base, 8 * base], start=1):
            stride = 1 if i == 1 else 2
            down = None if stride == 1 else nn.Sequential(nn.Conv2d(c_in, c, 1, stride, bias=False), nn.BatchNorm2d(c))
            setattr(self, f"layer{i}", nn.Sequential(BasicBlock(c_in, c, stride, down), BasicBlock(c, c)))
            c_in = c
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(dropout), nn.Linear(c_in, n_classes))
        for m in self.modules():   # same init as torchvision's ResNet
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")

    def forward(self, x):
        x = self.stem(x)
        x = self.layer4(self.layer3(self.layer2(self.layer1(x))))
        return self.head(x)


rs_hists, rs_rows = {}, []
for cfg in RESNET18_GRID:
    name = f"resnet18_b{cfg['base']}_d{cfg['dropout']:g}"
    model, hist, ckpt = fit(name, lambda: ResNet18(**cfg), cfg, **RESNET18_TRAIN)
    rs_hists[name] = hist
    kept = hist[hist.best].iloc[0]
    if config_done(name, cfg) is None or ckpt.stat().st_mtime > (ALL_DIR / name / "metrics.json").stat().st_mtime:   # new or retrained
        record_config(name, "resnet18_scratch", "torch", model.eval(), cfg, {sp: predict(model, sp) for sp in ("val", "test")}, kept.train_f1)
    rs_rows.append({**cfg, "name": name, "epoch": int(kept.epoch), "epochs_run": len(hist),
                    "stopped": "early stop" if len(hist) < RESNET18_TRAIN["max_epochs"] else "epoch limit",
                    "train_f1": kept.train_f1, "val_f1": kept.val_f1, "sec_per_epoch": hist.sec.mean(), "ckpt": str(ckpt)})
    del model
rs_grid = pd.DataFrame(rs_rows)
rs_grid["config"] = [f"b{b} d{d:g}" for b, d in zip(rs_grid.base, rs_grid.dropout)]
display(rs_grid.drop(columns="ckpt"))
plot_learning_curves(rs_hists, "ResNet-18 from scratch: learning curves", "resnet18_learning_curves")
plot_grid_summary(rs_grid, "config", "ResNet-18 scratch: train vs val macro-F1 by width / dropout", "resnet18_grid")

best_rs = rs_grid.loc[rs_grid.val_f1.idxmax()]
CHOSEN["resnet18_scratch"] = best_rs["name"]
rs_model = ResNet18(int(best_rs.base), float(best_rs.dropout)).to(DEVICE)
rs_model.load_state_dict(torch.load(best_rs.ckpt, map_location=DEVICE)); rs_model.eval()
print("chosen:", best_rs["name"])
report_model("resnet18_scratch", *config_scores(best_rs["name"]),
             {"base": int(best_rs.base), "dropout": float(best_rs.dropout), "epoch": int(best_rs.epoch)})
save_final("resnet18_scratch", "torch", rs_model, {"arch": "resnet18", "base": int(best_rs.base), "dropout": float(best_rs.dropout)})

## 8. Model comparison on this dataset

In [ ]:
MODELS = ["siglip_zeroshot", "bovw_svm", "effnet_b0", "resnet18_scratch"]
cols = ["top1", "top3", "top5", "macro_precision", "macro_recall", "macro_f1", "mAP"]
summary = pd.DataFrame({m: RESULTS[m]["test"] for m in MODELS}).T[cols]
summary.insert(0, "val_macro_f1", [RESULTS[m]["val"]["macro_f1"] for m in summary.index])
display(summary.style.highlight_max(axis=0, color="#cde4ff").format("{:.3f}"))
summary.to_csv(OUT / "summary_test.csv")

fig, ax = plt.subplots(figsize=(10, 4))
summary[["top1", "macro_f1", "mAP"]].plot.bar(ax=ax, rot=0)
ax.set_ylim(0, 1); ax.set_title(f"{DATASET}: test metrics by model"); ax.legend(loc="lower right")
plt.tight_layout(); savefig(fig, "model_comparison"); plt.show()

In [ ]:
# every config tried (read from all_models/, nothing is recomputed)
FAMILY = {"siglip_zeroshot": "SigLIP zero-shot", "bovw_svm": "SIFT BoVW + SVM", "effnet_b0": "EfficientNet-B0", "resnet18_scratch": "ResNet-18 scratch"}
all_cfg = pd.DataFrame([{"family": FAMILY[r["family"]], "model": n, "chosen": CHOSEN.get(r["family"]) == n, "train_f1": r["train_f1"],
                         "val_macro_f1": r["val"]["macro_f1"], **{f"test_{k}": v for k, v in r["test"].items() if k != "n"}}
                        for n, r in CONFIGS.items()])
all_cfg.to_csv(OUT / "all_configs_test.csv", index=False)
num = [c for c in all_cfg.columns if c not in ("family", "model", "chosen")]
display(all_cfg.style.format({c: "{:.3f}" for c in num}, na_rep="–")
        .apply(lambda r: ["font-weight: bold" if r.chosen else ""] * len(r), axis=1))

fig, ax = plt.subplots(figsize=(8, 0.32 * len(all_cfg) + 1.2))
fam = {f: f"C{j}" for j, f in enumerate(all_cfg.family.unique())}
ax.barh(all_cfg.model, all_cfg.test_macro_f1, color=[fam[f] for f in all_cfg.family],
        edgecolor=["black" if c else "none" for c in all_cfg.chosen], linewidth=1.5)
for y, (v, c) in enumerate(zip(all_cfg.test_macro_f1, all_cfg.chosen)):
    ax.text(v + 0.005, y, f"{v:.3f}" + ("  (chosen on val)" if c else ""), va="center", fontsize=8)
ax.invert_yaxis(); ax.set_xlim(0, 1.15); ax.set_xlabel("test macro-F1")
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in fam.values()], labels=list(fam), fontsize=8, loc="lower right")
ax.set_title(f"{DATASET}: test macro-F1 of every config tried (black edge = chosen on val)")
plt.tight_layout(); savefig(fig, "all_configs_test"); plt.show()

## 9. Grad-CAM: where the CNNs look
Grad-CAM weights the last conv block's feature maps by the gradient of the predicted class, giving a heat map of the regions that drove the
prediction. Same fixed test images (one per class) in every dataset, so 01 / 02 / 03 maps can be put side by side.

In [ ]:
class GradCAM:
    def __init__(self, model, layer):
        self.model = model
        self.handle = layer.register_forward_hook(self._hook)

    def _hook(self, module, inp, out):
        self.acts = out
        out.register_hook(lambda g: setattr(self, "grads", g))

    def __call__(self, x):
        self.model.eval()
        x = x.to(DEVICE).requires_grad_(True)   # makes the hook fire even when the backbone is frozen
        logits = self.model(x)
        cls = logits.argmax(1)
        self.model.zero_grad(set_to_none=True)
        logits.gather(1, cls[:, None]).sum().backward()
        A, G = self.acts.detach(), self.grads.detach()
        cam = F.relu((G.mean((2, 3), keepdim=True) * A).sum(1, keepdim=True))
        cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[:, 0]
        cam = (cam - cam.amin((1, 2), keepdim=True)) / (cam.amax((1, 2), keepdim=True) - cam.amin((1, 2), keepdim=True) + 1e-8)
        return cam.cpu().numpy(), logits.detach().softmax(1).cpu().numpy()

    def remove(self):
        self.handle.remove()


CAM_MODELS = {"EfficientNet-B0": (en_model, en_model.features[-1]), "ResNet-18": (rs_model, rs_model.layer4)}
TEST_IDX = {r: i for i, r in enumerate(splits["test"].relpath)}


def gradcam_grid(frame, title, name):
    """One row per test image (frame needs relpath, label, y): the image, then each CNN's heat map and prediction."""
    cams = {m: GradCAM(model, layer) for m, (model, layer) in CAM_MODELS.items()}
    fig, axes = plt.subplots(len(frame), 1 + len(cams), figsize=(3 * (1 + len(cams)), 3.1 * len(frame)), squeeze=False)
    try:
        for row, (_, r) in zip(axes, frame.iterrows()):
            img = Image.fromarray(IMAGES["test"][TEST_IDX[r.relpath]])
            shown = np.asarray(img.resize((IMG_SIZE, IMG_SIZE)))
            row[0].imshow(shown); row[0].set_title(f"true: {r.label}", fontsize=9)
            for ax, (m, cam) in zip(row[1:], cams.items()):
                heat, prob = cam(eval_tf(img)[None])
                p = prob[0].argmax()
                ax.imshow(shown); ax.imshow(heat[0], cmap="jet", alpha=0.45)
                ax.set_title(f"{m}: {CLASSES[p]} ({prob[0, p]:.2f})", fontsize=9, color="green" if p == r.y else "red")
            for ax in row: ax.axis("off")
    finally:
        for c in cams.values(): c.remove()
    finish(fig, title, name)


fixed = splits["test"].groupby("label", group_keys=False).sample(1, random_state=SEED).reset_index(drop=True)
fixed[["relpath"]].to_csv(OUT / "gradcam_fixed_images.csv", index=False)
gradcam_grid(fixed, f"{DATASET}: Grad-CAM, fixed test images (same in 01-03)", "gradcam_fixed")

## 10. Failure analysis: false negatives and why they fail
For each chosen model, on the test split:
- **False negatives per class**: how many images of each class were missed (recall loss) and which class took them.
- **Top confusion pairs**, with example images (true / predicted / confidence / 2nd choice).
- **Confident vs uncertain errors**: very confident mistakes are often look-alike dishes or label noise; low-confidence ones are borderline.
- **Why tags**, measured rather than guessed, for every wrong image:
  `dark` (mean grey of the original photo < 80, the pipeline's dark level), `low_contrast` (grey std in the lowest 25% of the test set),
  `gsam_fallback` / `small_mask` (dataset 03 only: segmentation failed, or the mask covers < 15% of the image), `confusable_pair`
  (true → predicted is one of the model's 3 most common confusions), `all_models_wrong` (all 4 models miss it: a hard or possibly mislabelled image).
- **Error rate with vs without each tag**: shows which conditions actually drive failures.

Saved per model to `results/<dataset>/<model>/failures.csv`. Brightness and contrast come from the `01_original` photo when it is on disk,
because darkness is a property of the photo, not of our preprocessing (CLAHE in 02 brightens it, and 03 adds a grey background).

In [ ]:
DARK_LEVEL, SMALL_MASK, N_PAIRS = 80, 0.15, 3

test = splits["test"][["relpath", "label", "y"]].copy()
photo_root = DATA_ROOT / "01_original" if (DATA_ROOT / "01_original").is_dir() else DATA_ROOT / DATASET
print("photo statistics from:", photo_root.name)


def grey_stats(path):
    g = np.asarray(Image.open(path).convert("L"), dtype=np.float32)
    return g.mean(), g.std()


test[["brightness", "contrast"]] = [grey_stats(photo_root / r) for r in test.relpath]
LOW_CONTRAST = np.percentile(test.contrast, 25)
TAG_RULES = {"dark": test.brightness < DARK_LEVEL, "low_contrast": test.contrast < LOW_CONTRAST}
if DATASET == "03_denoise_clahe_gsam" and GSAM_STATS is not None:
    g = GSAM_STATS.loc[test.relpath]
    test["mask_area"], test["gsam_fallback"] = g.mask_area.values, g.fallback.values.astype(bool)
    TAG_RULES["gsam_fallback"] = test.gsam_fallback
    TAG_RULES["small_mask"] = (test.mask_area < SMALL_MASK) & ~test.gsam_fallback
print(f"low_contrast threshold: grey std < {LOW_CONTRAST:.1f};", {t: int(v.sum()) for t, v in TAG_RULES.items()}, "test images per tag")

PREDS = {}
for m in MODELS:
    p = pd.read_csv(OUT / m / "predictions_test.csv")
    assert (p.relpath.values == test.relpath.values).all(), f"{m}: test order differs"
    PREDS[m] = p
ALL_WRONG = np.all([(p.y_pred != p.y_true).values for p in PREDS.values()], axis=0)
print(f"{ALL_WRONG.sum()} test images are wrong for all {len(PREDS)} models")


def confidence(scores):
    """Probability of the top class for probabilistic models; for the SVM (decision values) the top-1 minus top-2 margin."""
    s = np.sort(scores, axis=1)
    is_prob = np.allclose(scores.sum(1), 1, atol=1e-3) and (scores >= 0).all()
    return (s[:, -1] if is_prob else s[:, -1] - s[:, -2]), ("probability" if is_prob else "score margin")


def failure_analysis(m):
    p = PREDS[m]
    S = p[[f"score_{c}" for c in CLASSES]].values
    y, pred = p.y_true.values, p.y_pred.values
    wrong = pred != y
    conf, conf_name = confidence(S)
    second = np.argsort(-S, axis=1)[:, 1]

    # false negatives per class
    fn = pd.DataFrame({"true": [CLASSES[i] for i in y[wrong]], "pred": [CLASSES[i] for i in pred[wrong]]})
    per_class = pd.DataFrame({"n_test": pd.Series([CLASSES[i] for i in y]).value_counts(),
                              "false_negatives": fn.true.value_counts()}).fillna(0).astype(int).reindex(CLASSES)
    per_class["recall"] = 1 - per_class.false_negatives / per_class.n_test
    taker = fn.groupby("true").pred.agg(lambda s: f"{s.value_counts().index[0]} ({s.value_counts().iloc[0]})")
    per_class["mostly_predicted_as"] = taker.reindex(CLASSES).fillna("–")
    per_class = per_class.sort_values("false_negatives", ascending=False)
    assert per_class.false_negatives.sum() == len(y) - (pred == y).sum()
    display(per_class)

    pairs = fn.value_counts(["true", "pred"]).head(N_PAIRS)
    top_pairs = set(pairs.index)

    # why tags for each wrong image
    rows = []
    for i in np.flatnonzero(wrong):
        t = CLASSES[y[i]], CLASSES[pred[i]]
        tags = [tag for tag, rule in TAG_RULES.items() if rule.iloc[i]]
        tags += ["confusable_pair"] * (t in top_pairs) + ["all_models_wrong"] * bool(ALL_WRONG[i])
        rows.append({"relpath": p.relpath[i], "true": t[0], "pred": t[1], "second": CLASSES[second[i]], conf_name: conf[i],
                     "brightness": test.brightness.iloc[i], "contrast": test.contrast.iloc[i],
                     **({"mask_area": test.mask_area.iloc[i]} if "mask_area" in test else {}),
                     "tags": ";".join(tags) or "untagged"})
    failures = pd.DataFrame(rows)
    failures.to_csv(OUT / m / "failures.csv", index=False)

    # error rate with vs without each condition
    eff = pd.DataFrame([{"tag": tag, "n_with": int(rule.sum()), "error_with": wrong[rule.values].mean() if rule.any() else np.nan,
                         "error_without": wrong[~rule.values].mean()} for tag, rule in TAG_RULES.items()])
    eff["ratio"] = eff.error_with / eff.error_without
    display(eff.style.format({"error_with": "{:.3f}", "error_without": "{:.3f}", "ratio": "{:.2f}"}))
    print("tag counts among wrong images:", failures.tags.str.split(";").explode().value_counts().to_dict())

    fig, axes = plt.subplots(1, 2, figsize=(13, 3.6))
    axes[0].barh(per_class.index[::-1], per_class.false_negatives[::-1])
    axes[0].set_title(f"{m}: false negatives per class (of {per_class.n_test.iloc[0]})"); axes[0].set_xlabel("missed test images")
    axes[1].hist([conf[~wrong], conf[wrong]], bins=20, label=["correct", "wrong"], color=["tab:green", "tab:red"], alpha=0.7)
    axes[1].set_xlabel(conf_name); axes[1].set_title(f"{m}: confidence, correct vs wrong"); axes[1].legend()
    plt.tight_layout(); savefig(fig, f"{m}_fn_confidence"); plt.show()

    # gallery: the top confusion pairs
    if pairs.empty:
        return failures, eff
    n_show = 6
    fig, axes = plt.subplots(len(pairs), n_show, figsize=(2.6 * n_show, 3.4 * len(pairs)), squeeze=False)
    for row, ((t, pr), cnt) in zip(axes, pairs.items()):
        idx = [i for i in np.flatnonzero(wrong) if CLASSES[y[i]] == t and CLASSES[pred[i]] == pr][:n_show]
        for ax in row: ax.axis("off")
        for ax, i in zip(row, idx):
            ax.imshow(IMAGES["test"][i])
            ax.set_title(f"T: {t}\nP: {pr} ({conf[i]:.2f})\n2nd: {CLASSES[second[i]]}", fontsize=8, color="red")
        row[0].text(-0.08, 0.5, f"{t} → {pr}\n({cnt}×)", transform=row[0].transAxes, ha="right", va="center", fontsize=9)
    finish(fig, f"{m}: top {len(pairs)} confusion pairs (test)", f"{m}_confusion_pairs")
    return failures, eff


FAILURES, TAG_EFFECT = {}, {}
for m in MODELS:
    print(f"\n========== {m} ==========")
    FAILURES[m], TAG_EFFECT[m] = failure_analysis(m)

In [ ]:
# which conditions drive errors, all models together (error rate with the tag / without it; > 1 = the condition hurts)
ratio = pd.DataFrame({m: e.set_index("tag").ratio for m, e in TAG_EFFECT.items()})
display(ratio.style.format("{:.2f}").background_gradient(cmap="Reds", axis=None))
ratio.to_csv(OUT / "failure_tag_ratio.csv")

# Grad-CAM on images both CNNs get wrong (or either, if there are too few)
both = (PREDS["effnet_b0"].y_pred != PREDS["effnet_b0"].y_true) & (PREDS["resnet18_scratch"].y_pred != PREDS["resnet18_scratch"].y_true)
pool = test[both.values] if both.sum() >= 3 else test[(PREDS["effnet_b0"].y_pred != PREDS["effnet_b0"].y_true).values]
if len(pool):
    gradcam_grid(pool.sample(min(6, len(pool)), random_state=SEED).reset_index(drop=True),
                 f"{DATASET}: Grad-CAM on failures: does the model look at the food?", "gradcam_failures")

## 11. Cross-dataset comparison: does segmentation help?
Reads `results/<dataset>/<model>/predictions_test.csv` for every dataset already run, so run this notebook for 01, 02 and 03 first
(the 03 run then shows everything). Pairs use the **same test images**:
- **McNemar test**: of the images only one dataset gets right, is the split more one-sided than chance?
- **Bootstrap 95% CI of Δ macro-F1** (B − A, 1,000 resamples): if the interval contains 0, the gap is not evidence.

| pair | what it measures |
|---|---|
| 01 → 02 | denoise + CLAHE |
| 02 → 03 | Grounded-SAM segmentation (the research question) |
| 01 → 03 | everything together |

**Flip analysis (02 → 03)**: test images that are right on 02 but wrong on 03 ("segmentation hurt") and the reverse ("helped"),
with the mask area and fallback for each, and side-by-side 02 / 03 images. This shows directly when the mask removed food.

In [ ]:
from scipy.stats import binomtest

DATASETS = ["01_original", "02_denoise_clahe", "03_denoise_clahe_gsam"]
PAIRS = [("01_original", "02_denoise_clahe"), ("02_denoise_clahe", "03_denoise_clahe_gsam"), ("01_original", "03_denoise_clahe_gsam")]


def load_preds(ds, model):
    p = OUT.parent / ds / model / "predictions_test.csv"
    return pd.read_csv(p).set_index("relpath").sort_index() if p.exists() else None


rows, per_ds = [], []
rng = np.random.default_rng(SEED)
for model in MODELS:
    preds = {ds: load_preds(ds, model) for ds in DATASETS}
    for ds, pr in preds.items():
        if pr is not None:
            per_ds.append({"model": model, "dataset": ds, "top1": (pr.y_true == pr.y_pred).mean(),
                           "macro_f1": f1_score(pr.y_true, pr.y_pred, average="macro")})
    for a, b in PAIRS:
        pa, pb = preds[a], preds[b]
        if pa is None or pb is None:
            continue
        assert pa.index.equals(pb.index), f"{a} and {b} have different test images"
        y, ya, yb = pa.y_true.values, pa.y_pred.values, pb.y_pred.values
        ca, cb = ya == y, yb == y
        n01, n10 = int((ca & ~cb).sum()), int((~ca & cb).sum())
        p_mc = binomtest(n01, n01 + n10, 0.5).pvalue if n01 + n10 else 1.0
        boot = []
        for _ in range(1000):
            i = rng.integers(0, len(y), len(y))
            boot.append(f1_score(y[i], yb[i], average="macro") - f1_score(y[i], ya[i], average="macro"))
        rows.append({"model": model, "A": a, "B": b, "acc_A": ca.mean(), "acc_B": cb.mean(), "delta_acc": cb.mean() - ca.mean(),
                     "delta_macro_f1": f1_score(y, yb, average="macro") - f1_score(y, ya, average="macro"),
                     "f1_ci_low": np.percentile(boot, 2.5), "f1_ci_high": np.percentile(boot, 97.5),
                     "only_A_right": n01, "only_B_right": n10, "mcnemar_p": p_mc})

if per_ds:
    pivot = pd.DataFrame(per_ds).pivot(index="model", columns="dataset", values="macro_f1")
    display(pivot.style.format("{:.3f}").background_gradient(cmap="Blues", axis=1))
    fig, ax = plt.subplots(figsize=(10, 4))
    pivot.reindex(columns=[d for d in DATASETS if d in pivot.columns]).plot.bar(ax=ax, rot=0)
    ax.set_ylabel("test macro-F1"); ax.set_ylim(0, 1); ax.set_title("macro-F1 by model and dataset")
    plt.tight_layout(); savefig(fig, "cross_dataset"); plt.show()
paired = pd.DataFrame(rows)
if rows:
    display(paired.style.format({c: "{:.3f}" for c in paired.columns if c not in ("model", "A", "B", "only_A_right", "only_B_right")}))
    paired.to_csv(OUT.parent / "paired_tests.csv", index=False)
else:
    print("only", sorted({r['dataset'] for r in per_ds}), "found so far; run the other datasets to compare")

In [ ]:
# flip analysis 02 -> 03: when does segmentation hurt or help?
A, B = "02_denoise_clahe", "03_denoise_clahe_gsam"
gs = pd.read_csv(META / "gsam_stats.csv").set_index("relpath") if (META / "gsam_stats.csv").exists() else None
flips = []
for model in MODELS:
    pa, pb = load_preds(A, model), load_preds(B, model)
    if pa is None or pb is None:
        continue
    ca, cb = (pa.y_pred == pa.y_true).values, (pb.y_pred == pb.y_true).values
    for i in np.flatnonzero(ca != cb):
        r = pa.index[i]
        flips.append({"model": model, "relpath": r, "label": pa.label.iloc[i], "effect": "hurt" if ca[i] else "helped",
                      "pred_02": pa.pred_label.iloc[i], "pred_03": pb.pred_label.iloc[i],
                      **({"mask_area": gs.mask_area[r], "gsam_fallback": bool(gs.fallback[r])} if gs is not None else {})})
    if len(paired):
        row = paired[(paired.model == model) & (paired.A == A) & (paired.B == B)].iloc[0]
        assert (ca != cb).sum() == row.only_A_right + row.only_B_right   # same counts as McNemar's off-diagonal

if not flips:
    print(f"run both {A} and {B} first")
else:
    flips = pd.DataFrame(flips)
    flips.to_csv(OUT.parent / "flips_02_03.csv", index=False)
    agg = {"n": ("relpath", "size")}
    if gs is not None:
        agg.update(median_mask_area=("mask_area", "median"), fallback_rate=("gsam_fallback", "mean"))
    summary_flips = flips.groupby(["model", "effect"]).agg(**agg).unstack("effect")
    display(summary_flips)
    if gs is not None:
        test_gs = gs[gs.split == "test"]
        print(f"for reference, all test images: median mask area {test_gs.mask_area.median():.3f}, fallback rate {test_gs.fallback.mean():.3f}")
    display(flips[flips.effect == "hurt"].label.value_counts().rename("hurt flips per class (all models)").to_frame().T)

    # side-by-side 02 / 03 of 'hurt' flips, if both image folders are on disk
    if (DATA_ROOT / A).is_dir() and (DATA_ROOT / B).is_dir():
        for model in MODELS:
            hurt = flips[(flips.model == model) & (flips.effect == "hurt")].head(4)
            if hurt.empty:
                continue
            fig, axes = plt.subplots(2, len(hurt), figsize=(3 * len(hurt), 6.4), squeeze=False)
            for j, (_, r) in enumerate(hurt.iterrows()):
                axes[0, j].imshow(Image.open(DATA_ROOT / A / r.relpath)); axes[0, j].set_title(f"02 ✓ {r.pred_02}", fontsize=9, color="green")
                note = f"\nmask {r.mask_area:.2f}{' fallback' if r.gsam_fallback else ''}" if gs is not None else ""
                axes[1, j].imshow(Image.open(DATA_ROOT / B / r.relpath)); axes[1, j].set_title(f"03 ✗ {r.pred_03}{note}", fontsize=9, color="red")
                for ax in axes[:, j]: ax.axis("off")
            finish(fig, f"{model}: right on 02, wrong after segmentation (03)", f"flips_hurt_{model}")

## 12. Checklist for the report
- [ ] All three datasets run in `full` mode with the same settings (only `DATASET` changed).
- [ ] Per model and dataset: test top-1 and macro P/R/F1 from `summary_test.csv`; the chosen config came from val.
- [ ] 02 → 03 Δ macro-F1 **with its bootstrap CI and McNemar p**; one seed only, so a CI that contains 0 is not evidence.
- [ ] Segmentation fallback rate for 03 (printed in section 1).
- [ ] Failure analysis: classes with the most false negatives, top confusion pairs, which tags raise the error rate.
- [ ] Flip analysis: how many images segmentation hurt vs helped, and whether the hurt ones have small masks or fallbacks.
- [ ] Every paper in `doc/references.md` checked (⚠️ → ✅) before it goes on a slide.